In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1. Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"train shape: {train.shape}, test shape: {test.shape}")

train["is_train"] = 1
test["is_train"] = 0
test["demand"] = np.nan

full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)


# 2. Time features
def parse_minutes(ts: str) -> int:
    h, m = ts.split(":")
    return int(h) * 60 + int(m)

full["minute_of_day"] = full["timestamp"].map(parse_minutes)
full["time_idx"] = full["day"] * 1440 + full["minute_of_day"]  # global time order
full["hour"] = full["minute_of_day"] // 60



full["tod_sin"] = np.sin(2 * np.pi * full["minute_of_day"] / 1440)
full["tod_cos"] = np.cos(2 * np.pi * full["minute_of_day"] / 1440)


full["is_morning_rush"] = full["hour"].between(7, 10).astype(int)
full["is_evening_rush"] = full["hour"].between(17, 20).astype(int)
full["is_night"] = ((full["hour"] >= 23) | (full["hour"] <= 4)).astype(int)



# 3. Decode geohash -> lat/lon (standard base32 geohash decoding, no
#    external dependency needed)


_BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
_DECODE_MAP = {c: i for i, c in enumerate(_BASE32)}


def decode_geohash(gh: str):
    lat_range = [-90.0, 90.0]
    lon_range = [-180.0, 180.0]
    is_lon = True
    for ch in gh:
        cd = _DECODE_MAP[ch]
        for bit in range(4, -1, -1):
            bit_val = (cd >> bit) & 1
            if is_lon:
                mid = (lon_range[0] + lon_range[1]) / 2
                if bit_val:
                    lon_range[0] = mid
                else:
                    lon_range[1] = mid
            else:
                mid = (lat_range[0] + lat_range[1]) / 2
                if bit_val:
                    lat_range[0] = mid
                else:
                    lat_range[1] = mid
            is_lon = not is_lon
    lat = (lat_range[0] + lat_range[1]) / 2
    lon = (lon_range[0] + lon_range[1]) / 2
    return lat, lon


geohash_unique = full["geohash"].unique()
geo_lookup = {g: decode_geohash(g) for g in geohash_unique}
full["lat"] = full["geohash"].map(lambda g: geo_lookup[g][0])
full["lon"] = full["geohash"].map(lambda g: geo_lookup[g][1])



full["geohash_id"] = full["geohash"].astype("category").cat.codes



# 4. Leakage-free "yesterday same time" feature
day48 = full[full["day"] == 48][["geohash", "minute_of_day", "demand"]].rename(
    columns={"demand": "demand_prev_day"}
)
full = full.merge(day48, on=["geohash", "minute_of_day"], how="left")
full.loc[full["day"] == 48, "demand_prev_day"] = np.nan


# 5. Global time-of-day demand pattern (learned from day 48), leave-one-out
#    for day-48 rows to avoid leakage

d48 = full[full["day"] == 48]
tod_stats = d48.groupby("minute_of_day")["demand"].agg(["sum", "count"])
full = full.merge(
    tod_stats.rename(columns={"sum": "_tod_sum", "count": "_tod_cnt"}),
    on="minute_of_day",
    how="left",
)

is_d48 = full["day"] == 48

loo_sum = full["_tod_sum"] - full["demand"].fillna(0)
loo_cnt = full["_tod_cnt"] - 1
full["tod_global_mean"] = np.where(
    is_d48 & (loo_cnt > 0),
    loo_sum / loo_cnt,
    full["_tod_sum"] / full["_tod_cnt"],
)
full.drop(columns=["_tod_sum", "_tod_cnt"], inplace=True)


# 6. Per-geohash expanding statistics using ONLY strictly earlier timestamps
#    (proper walk-forward / no-leakage rolling features)

full.sort_values(["geohash", "time_idx"], inplace=True)
full.reset_index(drop=True, inplace=True)

known = full["demand"].fillna(0.0)
is_known = full["demand"].notna().astype(float)

grp = full.groupby("geohash", sort=False)
cum_sum = grp.apply(lambda s: known.loc[s.index].cumsum()).reset_index(level=0, drop=True)
cum_cnt = grp.apply(lambda s: is_known.loc[s.index].cumsum()).reset_index(level=0, drop=True)


cum_sum_prev = cum_sum.groupby(full["geohash"]).shift(1)
cum_cnt_prev = cum_cnt.groupby(full["geohash"]).shift(1)

full["geo_expanding_mean"] = cum_sum_prev / cum_cnt_prev
full["geo_expanding_count"] = cum_cnt_prev.fillna(0)


demand_known_series = full["demand"].where(full["demand"].notna())
full["_last_val"] = demand_known_series.groupby(full["geohash"]).ffill()
full["geo_last_demand"] = full.groupby("geohash")["_last_val"].shift(1)

full["_last_time_idx"] = full["time_idx"].where(full["demand"].notna())
full["_last_time_idx_ffill"] = full.groupby("geohash")["_last_time_idx"].ffill()
full["geo_time_since_last"] = full["time_idx"] - full.groupby("geohash")[
    "_last_time_idx"
].shift(0).groupby(full["geohash"]).ffill().groupby(full["geohash"]).shift(1)

last_idx_shifted = full.groupby("geohash")["_last_time_idx_ffill"].shift(1)
full["geo_time_since_last"] = full["time_idx"] - last_idx_shifted
full.drop(columns=["_last_val", "_last_time_idx", "_last_time_idx_ffill"], inplace=True)


full["geo_roll4_mean"] = (
    demand_known_series.groupby(full["geohash"])
    .apply(lambda s: s.rolling(4, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)
full["geo_roll4_mean"] = full.groupby("geohash")["geo_roll4_mean"].shift(1)


full.sort_values("Index", inplace=True)
full = full.sort_values(["is_train", "Index"], ascending=[False, True])


# 7. Categorical feature cleanup

cat_cols = ["RoadType", "LargeVehicles", "Landmarks", "Weather"]
for c in cat_cols:
    full[c] = full[c].astype("category")
full["NumberofLanes"] = full["NumberofLanes"].astype("category")


# 8. Final feature list

feature_cols = [
    "geohash_id", "lat", "lon",
    "day", "minute_of_day", "hour", "tod_sin", "tod_cos",
    "is_morning_rush", "is_evening_rush", "is_night",
    "RoadType", "NumberofLanes", "LargeVehicles", "Landmarks",
    "Temperature", "Weather",
    "demand_prev_day", "tod_global_mean",
    "geo_expanding_mean", "geo_expanding_count",
    "geo_last_demand", "geo_time_since_last", "geo_roll4_mean",
]

train_df = full[full["is_train"] == 1].copy()
test_df = full[full["is_train"] == 0].copy()

X = train_df[feature_cols]
y = train_df["demand"]
X_test = test_df[feature_cols]

cat_features = ["RoadType", "NumberofLanes", "LargeVehicles", "Landmarks", "Weather"]

print(f"\nFeature matrix: {X.shape}, Test matrix: {X_test.shape}")


# 9. Time-respecting validation split
#    (train on day48 + early day49, validate on the LAST slots of day 49
#    inside train -> mimics the real train/test time gap)

val_mask = train_df["day"] == 49
X_tr, y_tr = X[~val_mask], y[~val_mask]
X_val, y_val = X[val_mask], y[val_mask]

print(f"Time-based split -> train: {X_tr.shape}, val: {X_val.shape}")

lgb_train = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_features)
lgb_val = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_features, reference=lgb_train)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_data_in_leaf": 30,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbosity": -1,
    "seed": RANDOM_STATE,
}

model_val = lgb.train(
    params,
    lgb_train,
    num_boost_round=3000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

val_pred = model_val.predict(X_val, num_iteration=model_val.best_iteration)
val_pred = np.clip(val_pred, 0, 1)
rmse = np.sqrt(np.mean((val_pred - y_val.values) ** 2))
mae = np.mean(np.abs(val_pred - y_val.values))
print(f"\nTime-based validation -> RMSE: {rmse:.5f}  MAE: {mae:.5f}")


imp = pd.DataFrame(
    {"feature": feature_cols, "gain": model_val.feature_importance(importance_type="gain")}
).sort_values("gain", ascending=False)
print("\nTop feature importances (gain):")
print(imp.head(15).to_string(index=False))


# 10. Final model: 5-fold CV bagging on ALL train data, predict test with
#     the average of the fold models (more robust than a single split)

best_rounds = max(model_val.best_iteration, 100)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr_f, y_tr_f = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va_f, y_va_f = X.iloc[va_idx], y.iloc[va_idx]

    dtr = lgb.Dataset(X_tr_f, label=y_tr_f, categorical_feature=cat_features)
    dva = lgb.Dataset(X_va_f, label=y_va_f, categorical_feature=cat_features, reference=dtr)

    fold_model = lgb.train(
        params,
        dtr,
        num_boost_round=int(best_rounds * 1.2),
        valid_sets=[dva],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    oof_preds[va_idx] = np.clip(
        fold_model.predict(X_va_f, num_iteration=fold_model.best_iteration), 0, 1
    )
    fold_rmse = np.sqrt(np.mean((oof_preds[va_idx] - y_va_f.values) ** 2))
    fold_rmses.append(fold_rmse)
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.5f} (best_iter={fold_model.best_iteration})")

    test_preds += np.clip(
        fold_model.predict(X_test, num_iteration=fold_model.best_iteration), 0, 1
    ) / kf.n_splits

print(f"\nOOF RMSE (5-fold): {np.sqrt(np.mean((oof_preds - y.values) ** 2)):.5f}")
print(f"Mean fold RMSE: {np.mean(fold_rmses):.5f} +/- {np.std(fold_rmses):.5f}")


